# Notebook 08 — Bootstrap Confidence Intervals

**Purpose:** Compute 95% bootstrap CIs for all key findings from Day 3 to make paper claims reviewer-defensible. Same protocol as 03d (B=1000 stratified resamples, percentile method, SEED=42).

## Seven CI targets

| # | Target | Method | Stratification |
|---|---|---|---|
| 1 | Calibration Brier/ECE per-class | Bootstrap B=1000 | By class |
| 2 | Krishna rank correlations | Bootstrap B=1000 | Per (ds, pair, K) |
| 3 | DNN stability advantage (mean Jaccard) | Bootstrap B=1000 | Per (ds, model) |
| 4 | SCTS Mondrian Pearson per dataset | Bootstrap B=1000 | Per (ds, model) |
| 5 | SCTS Normal-vs-U2R per-class gap | Bootstrap B=1000 | By true class |
| 6 | GREEN vs RED Pearson gap | Bootstrap B=1000 | Within flag group, per (ds, model) |
| 7 | NSL R2L RED-flag rate | Wilson binomial CI | (closed form) |

## Protocol

- B=1000 bootstrap resamples per target
- 95% CI via percentile method (2.5/97.5)
- SEED=42 (matches 03d)
- Each iteration uses an independent RNG seeded as `SEED + b` for parallel-friendliness
- Wilson score interval for binomial proportion (Target 7)

## Outputs

- `results/tables/bootstrap_cis_calibration.csv` (Target 1)
- `results/tables/bootstrap_cis_krishna.csv` (Target 2)
- `results/tables/bootstrap_cis_stability.csv` (Target 3)
- `results/tables/bootstrap_cis_scts_pearson.csv` (Target 4)
- `results/tables/bootstrap_cis_scts_class_gap.csv` (Target 5)
- `results/tables/bootstrap_cis_flag_pearson.csv` (Target 6)
- `results/tables/bootstrap_cis_r2l_flag_rate.csv` (Target 7)
- `results/tables/bootstrap_cis_summary.json` (aggregate)

**Estimated time:** 30-60 minutes total. Per-target compute is dominated by Target 1 (B=1000 × 3 datasets × 6 models × 5 classes ≈ 90k Brier/ECE computations) and Target 4 (B=1000 × 18 models × Pearson).

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

print(f'✓ Ready in: {os.getcwd()}')

Mounted at /content/drive
✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
import numpy as np
import pandas as pd
import json, time
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

SEED = 42
B = 1000  # bootstrap iterations
ALPHA = 0.05  # for 95% CI

DATASETS = ['nsl_kdd_v2', 'unsw_nb15_v2', 'cic_ids2017_v2']
ARCHITECTURES = ['rf', 'xgb', 'dnn']
VARIANTS = ['5class_cw', '5class_smote']
MODELS_PER_DATASET = [f'{a}_{v}' for v in VARIANTS for a in ARCHITECTURES]
CLASS_NAMES_5 = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']

TABLES_DIR = Path(REPO) / 'results' / 'tables'

print(f'Bootstrap: B={B}, 95% CI, SEED={SEED}')
print(f'Datasets: {len(DATASETS)}, Models per dataset: {len(MODELS_PER_DATASET)}')

Bootstrap: B=1000, 95% CI, SEED=42
Datasets: 3, Models per dataset: 6


## 2. Bootstrap helper functions

In [3]:
def bootstrap_ci(values, statistic_fn, B=1000, alpha=0.05, seed=42):
    """Generic non-stratified bootstrap.

    values: array-like, the data to resample
    statistic_fn: callable taking resampled values, returns scalar
    Returns (point, lower, upper) at (1-alpha)% CI.
    """
    values = np.asarray(values)
    n = len(values)
    rng = np.random.default_rng(seed)
    samples = np.empty(B)
    for b in range(B):
        idx = rng.integers(0, n, size=n)
        samples[b] = statistic_fn(values[idx])
    point = statistic_fn(values)
    lo = np.quantile(samples, alpha / 2)
    hi = np.quantile(samples, 1 - alpha / 2)
    return float(point), float(lo), float(hi)

def bootstrap_ci_paired(values1, values2, statistic_fn, B=1000, alpha=0.05, seed=42):
    """Paired bootstrap: resample indices, compute statistic on both arrays.
    statistic_fn: callable(v1, v2) -> scalar
    """
    values1 = np.asarray(values1)
    values2 = np.asarray(values2)
    assert len(values1) == len(values2), 'paired bootstrap requires equal lengths'
    n = len(values1)
    rng = np.random.default_rng(seed)
    samples = np.empty(B)
    for b in range(B):
        idx = rng.integers(0, n, size=n)
        samples[b] = statistic_fn(values1[idx], values2[idx])
    point = statistic_fn(values1, values2)
    lo = np.quantile(samples, alpha / 2)
    hi = np.quantile(samples, 1 - alpha / 2)
    return float(point), float(lo), float(hi)

def bootstrap_ci_stratified(values, strata, statistic_fn, B=1000, alpha=0.05, seed=42):
    """Stratified bootstrap: preserves stratum sizes per resample.
    values, strata: arrays of equal length
    """
    values = np.asarray(values)
    strata = np.asarray(strata)
    unique_strata = np.unique(strata)
    strata_idx = {s: np.where(strata == s)[0] for s in unique_strata}
    rng = np.random.default_rng(seed)
    samples = np.empty(B)
    for b in range(B):
        # Resample within each stratum
        idx_parts = [rng.choice(strata_idx[s], size=len(strata_idx[s]), replace=True) for s in unique_strata]
        idx = np.concatenate(idx_parts)
        samples[b] = statistic_fn(values[idx], strata[idx])
    point = statistic_fn(values, strata)
    lo = np.quantile(samples, alpha / 2)
    hi = np.quantile(samples, 1 - alpha / 2)
    return float(point), float(lo), float(hi)

def wilson_ci(successes, n, alpha=0.05):
    """Wilson score interval for binomial proportion.
    Better than Clopper-Pearson near boundaries (0/1).
    """
    from scipy.stats import norm
    if n == 0:
        return float('nan'), float('nan'), float('nan')
    z = norm.ppf(1 - alpha / 2)
    p = successes / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return float(p), float(center - half), float(center + half)

print('✓ Bootstrap helpers loaded')

✓ Bootstrap helpers loaded


## 3. Target 1: Calibration Brier and ECE per-class CIs

In [4]:
def brier_score(y_true, y_proba_onehot):
    """Multi-class Brier score = mean squared difference between predicted probs and one-hot truth."""
    return float(((y_proba_onehot - 1.0 * (y_true[:, None] == np.arange(y_proba_onehot.shape[1])[None, :])) ** 2).sum(axis=1).mean())

def ece(y_true, y_proba, n_bins=10):
    """Expected Calibration Error (multi-class, top-label)."""
    confidences = y_proba.max(axis=1)
    predictions = y_proba.argmax(axis=1)
    accuracies = (predictions == y_true).astype(float)
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece_val = 0.0
    n = len(y_true)
    for i in range(n_bins):
        mask = (confidences >= bin_edges[i]) & (confidences < bin_edges[i + 1])
        if i == n_bins - 1:
            mask = (confidences >= bin_edges[i]) & (confidences <= bin_edges[i + 1])
        if mask.sum() == 0:
            continue
        bin_acc = accuracies[mask].mean()
        bin_conf = confidences[mask].mean()
        ece_val += (mask.sum() / n) * abs(bin_acc - bin_conf)
    return float(ece_val)

calib_records = []
t0 = time.time()
for ds in DATASETS:
    y_test = np.load(f'{REPO}/data/processed/{ds}/y_test_5class.npy')
    for model_name in MODELS_PER_DATASET:
        proba = np.load(Path(REPO) / 'calibrators' / ds / f'{model_name}_test_proba_calibrated.npy')

        # Per-class Brier (one-vs-rest) with CI
        for c in range(5):
            y_c = (y_test == c).astype(int)
            p_c = proba[:, c]
            # Brier for class c: mean((p - y)^2)
            point, lo, hi = bootstrap_ci_paired(y_c, p_c, lambda y, p: float(((p - y) ** 2).mean()), B=B, alpha=ALPHA, seed=SEED)
            calib_records.append({
                'dataset': ds, 'model': model_name, 'class': CLASS_NAMES_5[c],
                'metric': 'brier_per_class',
                'point': point, 'lo_95': lo, 'hi_95': hi, 'n': len(y_c),
            })

        # Overall ECE (top-label, 10 bins) — single value per model
        n_samples = len(y_test)
        rng = np.random.default_rng(SEED)
        ece_boot = np.empty(B)
        for b in range(B):
            idx = rng.integers(0, n_samples, size=n_samples)
            ece_boot[b] = ece(y_test[idx], proba[idx])
        point = ece(y_test, proba)
        lo = np.quantile(ece_boot, ALPHA / 2)
        hi = np.quantile(ece_boot, 1 - ALPHA / 2)
        calib_records.append({
            'dataset': ds, 'model': model_name, 'class': 'OVERALL',
            'metric': 'ece_top_label',
            'point': float(point), 'lo_95': float(lo), 'hi_95': float(hi), 'n': n_samples,
        })
    print(f'  {ds}: done ({(time.time()-t0)/60:.1f} min elapsed)')

df_calib = pd.DataFrame(calib_records)
df_calib.to_csv(TABLES_DIR / 'bootstrap_cis_calibration.csv', index=False)
print(f'✓ Saved bootstrap_cis_calibration.csv ({len(df_calib)} rows, {(time.time()-t0)/60:.1f} min)')

# Show summary
print('\nPer-class Brier summary (mean across 6 models per dataset):')
summary = df_calib[df_calib['metric'] == 'brier_per_class'].groupby(['dataset', 'class']).agg({
    'point': 'mean', 'lo_95': 'mean', 'hi_95': 'mean',
}).round(4)
print(summary)

  nsl_kdd_v2: done (0.7 min elapsed)
  unsw_nb15_v2: done (3.0 min elapsed)
  cic_ids2017_v2: done (4.2 min elapsed)
✓ Saved bootstrap_cis_calibration.csv (108 rows, 4.2 min)

Per-class Brier summary (mean across 6 models per dataset):
                        point   lo_95   hi_95
dataset        class                         
cic_ids2017_v2 DoS     0.0053  0.0049  0.0057
               Normal  0.0091  0.0086  0.0097
               Probe   0.0029  0.0026  0.0032
               R2L     0.0010  0.0008  0.0012
               U2R     0.0001  0.0000  0.0002
nsl_kdd_v2     DoS     0.0649  0.0619  0.0682
               Normal  0.1943  0.1893  0.1997
               Probe   0.0419  0.0395  0.0442
               R2L     0.1172  0.1128  0.1214
               U2R     0.0032  0.0025  0.0039
unsw_nb15_v2   DoS     0.0462  0.0451  0.0472
               Normal  0.1179  0.1162  0.1196
               Probe   0.0299  0.0289  0.0309
               R2L     0.1507  0.1492  0.1523
               U2R     0.007

## 4. Target 2: Krishna rank correlations

In [5]:
df_krishna = pd.read_csv(TABLES_DIR / 'krishna_agreement_canonical.csv')
print(f'Krishna data: {len(df_krishna)} rows, columns: {df_krishna.columns.tolist()}')
print(f'Sample row:\n{df_krishna.iloc[0]}')

# The Krishna CSV has per-sample agreement scores. Need to find the column structure.
# We expect to find per-(dataset, pair, K) rows and compute mean RC or similar with CI.

# Identify what to bootstrap. From v9 §15 / v10 §16: mean RC across 1000 samples per (ds, pair, K).
krishna_records = []
t0 = time.time()

# Group by (dataset, pair, K) and bootstrap the rank correlation column
# Need to identify the right column for rank correlation
rc_col = None
for c in df_krishna.columns:
    if 'rank' in c.lower() or 'spearman' in c.lower() or 'kendall' in c.lower() or c.lower() == 'rc' or 'agreement' in c.lower():
        print(f'Candidate rank correlation column: {c}')
        rc_col = c if rc_col is None else rc_col
print(f'Using column: {rc_col}')

# If we have spearman_rho and kendall_tau columns specifically, use both
metric_cols = [c for c in df_krishna.columns if c.lower() in ('spearman_rho', 'kendall_tau', 'spearman', 'kendall', 'rc')]
if not metric_cols and rc_col:
    metric_cols = [rc_col]
print(f'Metric columns to bootstrap: {metric_cols}')

# Group columns: dataset, pair, K (or 'k')
group_cols = ['dataset', 'pair']
if 'K' in df_krishna.columns:
    group_cols.append('K')
elif 'k' in df_krishna.columns:
    group_cols.append('k')

for keys, g in df_krishna.groupby(group_cols):
    for metric in metric_cols:
        vals = g[metric].dropna().values
        if len(vals) < 5:
            continue
        point, lo, hi = bootstrap_ci(vals, lambda v: float(np.mean(v)), B=B, alpha=ALPHA, seed=SEED)
        rec = dict(zip(group_cols, keys if isinstance(keys, tuple) else (keys,)))
        rec.update({'metric': metric, 'point': point, 'lo_95': lo, 'hi_95': hi, 'n': len(vals)})
        krishna_records.append(rec)

df_krishna_cis = pd.DataFrame(krishna_records)
df_krishna_cis.to_csv(TABLES_DIR / 'bootstrap_cis_krishna.csv', index=False)
print(f'\n✓ Saved bootstrap_cis_krishna.csv ({len(df_krishna_cis)} rows, {(time.time()-t0)/60:.1f} min)')
print('\nMean Krishna CI per dataset:')
print(df_krishna_cis.groupby('dataset')[['point', 'lo_95', 'hi_95']].mean().round(3))

Krishna data: 324 rows, columns: ['dataset', 'variant', 'pair', 'k', 'class', 'FA', 'RA', 'SA', 'SRA', 'RC', 'PRA', 'class_idx', 'n_samples']
Sample row:
dataset      nsl_kdd_v2
variant       5class_cw
pair             rf-xgb
k                     5
class         aggregate
FA                0.649
RA               0.2508
SA                  1.0
SRA              0.2508
RC               0.5978
PRA              0.7514
class_idx           NaN
n_samples           NaN
Name: 0, dtype: object
Candidate rank correlation column: RC
Using column: RC
Metric columns to bootstrap: ['RC']

✓ Saved bootstrap_cis_krishna.csv (27 rows, 0.0 min)

Mean Krishna CI per dataset:
                point  lo_95  hi_95
dataset                            
cic_ids2017_v2  0.189  0.057  0.302
nsl_kdd_v2      0.271  0.197  0.347
unsw_nb15_v2    0.280  0.217  0.346


## 5. Target 3: DNN stability advantage (mean Jaccard, per-model with CI)

In [6]:
df_stab = pd.read_csv(TABLES_DIR / 'stability_v2_per_sample_jaccard.csv')
print(f'Loaded stability: {len(df_stab)} rows')

# Per (dataset, model, perturbation): mean Jaccard with CI
# Then per (dataset, model): mean Jaccard ACROSS perturbations
# Then check DNN advantage: is DNN mean Jaccard > tree mean Jaccard?

stab_records = []
t0 = time.time()

for (ds, model), g in df_stab.groupby(['dataset', 'model']):
    # Bootstrap mean Jaccard across samples, pooling all perturbations
    vals = g['jaccard_top10'].values
    point, lo, hi = bootstrap_ci(vals, lambda v: float(np.mean(v)), B=B, alpha=ALPHA, seed=SEED)
    arch = 'rf' if 'rf' in model else ('xgb' if 'xgb' in model else 'dnn')
    stab_records.append({
        'dataset': ds, 'model': model, 'architecture': arch,
        'mean_jaccard': point, 'lo_95': lo, 'hi_95': hi, 'n': len(vals),
    })

df_stab_cis = pd.DataFrame(stab_records)

# DNN advantage: for each (dataset, variant=cw|smote), compare DNN to RF and XGB
advantage_records = []
for ds in DATASETS:
    for variant in ['5class_cw', '5class_smote']:
        sub = df_stab_cis[(df_stab_cis['dataset'] == ds) & (df_stab_cis['model'].str.contains(variant))]
        if len(sub) != 3:
            continue
        dnn_row = sub[sub['architecture'] == 'dnn'].iloc[0]
        rf_row = sub[sub['architecture'] == 'rf'].iloc[0]
        xgb_row = sub[sub['architecture'] == 'xgb'].iloc[0]

        # Use paired bootstrap on the per-sample Jaccards
        # Need to align samples by sample_position+perturbation
        dnn_vals = df_stab[(df_stab['dataset'] == ds) & (df_stab['model'] == dnn_row['model'])].sort_values(['sample_position', 'perturbation'])['jaccard_top10'].values
        rf_vals = df_stab[(df_stab['dataset'] == ds) & (df_stab['model'] == rf_row['model'])].sort_values(['sample_position', 'perturbation'])['jaccard_top10'].values
        xgb_vals = df_stab[(df_stab['dataset'] == ds) & (df_stab['model'] == xgb_row['model'])].sort_values(['sample_position', 'perturbation'])['jaccard_top10'].values

        # DNN vs RF advantage
        if len(dnn_vals) == len(rf_vals):
            point, lo, hi = bootstrap_ci_paired(dnn_vals, rf_vals, lambda d, r: float(np.mean(d) - np.mean(r)), B=B, alpha=ALPHA, seed=SEED)
            advantage_records.append({
                'dataset': ds, 'variant': variant, 'comparison': 'dnn_minus_rf',
                'point': point, 'lo_95': lo, 'hi_95': hi, 'n': len(dnn_vals),
                'dnn_advantage': point > 0,
                'significant_advantage': lo > 0,
            })

        # DNN vs XGB advantage
        if len(dnn_vals) == len(xgb_vals):
            point, lo, hi = bootstrap_ci_paired(dnn_vals, xgb_vals, lambda d, x: float(np.mean(d) - np.mean(x)), B=B, alpha=ALPHA, seed=SEED)
            advantage_records.append({
                'dataset': ds, 'variant': variant, 'comparison': 'dnn_minus_xgb',
                'point': point, 'lo_95': lo, 'hi_95': hi, 'n': len(dnn_vals),
                'dnn_advantage': point > 0,
                'significant_advantage': lo > 0,
            })

df_advantage = pd.DataFrame(advantage_records)
df_stab_cis.to_csv(TABLES_DIR / 'bootstrap_cis_stability.csv', index=False)
df_advantage.to_csv(TABLES_DIR / 'bootstrap_cis_stability_advantage.csv', index=False)

print(f'✓ Saved bootstrap_cis_stability.csv ({len(df_stab_cis)} rows)')
print(f'✓ Saved bootstrap_cis_stability_advantage.csv ({len(df_advantage)} rows, {(time.time()-t0)/60:.1f} min)')
print(f'\nDNN advantage cases (point > 0): {df_advantage["dnn_advantage"].sum()}/{len(df_advantage)}')
print(f'Significant DNN advantages (CI lo > 0): {df_advantage["significant_advantage"].sum()}/{len(df_advantage)}')
print(df_advantage[['dataset', 'variant', 'comparison', 'point', 'lo_95', 'hi_95', 'significant_advantage']].to_string(index=False))

Loaded stability: 54000 rows
✓ Saved bootstrap_cis_stability.csv (18 rows)
✓ Saved bootstrap_cis_stability_advantage.csv (12 rows, 0.0 min)

DNN advantage cases (point > 0): 9/12
Significant DNN advantages (CI lo > 0): 9/12
       dataset      variant    comparison     point     lo_95     hi_95  significant_advantage
    nsl_kdd_v2    5class_cw  dnn_minus_rf -0.099870 -0.106852 -0.092756                  False
    nsl_kdd_v2    5class_cw dnn_minus_xgb -0.092408 -0.099073 -0.086004                  False
    nsl_kdd_v2 5class_smote  dnn_minus_rf  0.040843  0.033965  0.048285                   True
    nsl_kdd_v2 5class_smote dnn_minus_xgb -0.019508 -0.026433 -0.012590                  False
  unsw_nb15_v2    5class_cw  dnn_minus_rf  0.090349  0.081922  0.097116                   True
  unsw_nb15_v2    5class_cw dnn_minus_xgb  0.087983  0.081098  0.094192                   True
  unsw_nb15_v2 5class_smote  dnn_minus_rf  0.040514  0.034039  0.047624                   True
  unsw_nb15_v2 5

## 6. Target 4: SCTS Mondrian Pearson per dataset

In [7]:
df_scts = pd.read_csv(TABLES_DIR / 'scts_v2_canonical.csv')
print(f'Loaded SCTS canonical: {len(df_scts)} rows')

pearson_records = []
t0 = time.time()

for (ds, model), g in df_scts.groupby(['dataset', 'model']):
    scts = g['scts'].values
    correct = g['correct'].values.astype(float)

    if scts.std() < 1e-9 or correct.std() < 1e-9:
        continue

    def pearson_fn(s, c):
        if s.std() < 1e-9 or c.std() < 1e-9:
            return np.nan
        return float(np.corrcoef(s, c)[0, 1])

    point, lo, hi = bootstrap_ci_paired(scts, correct, pearson_fn, B=B, alpha=ALPHA, seed=SEED)
    pearson_records.append({
        'dataset': ds, 'model': model,
        'pearson': point, 'lo_95': lo, 'hi_95': hi, 'n': len(scts),
    })

df_pearson = pd.DataFrame(pearson_records)
df_pearson.to_csv(TABLES_DIR / 'bootstrap_cis_scts_pearson.csv', index=False)
print(f'✓ Saved bootstrap_cis_scts_pearson.csv ({len(df_pearson)} rows, {(time.time()-t0)/60:.1f} min)')

# Per-dataset summary
print('\nPer-dataset Pearson (mean across 6 models, with bootstrap CI for the mean across models):')
for ds in DATASETS:
    sub = df_pearson[df_pearson['dataset'] == ds]
    if len(sub) == 0:
        continue
    # Bootstrap the model-level mean
    point, lo, hi = bootstrap_ci(sub['pearson'].values, lambda v: float(np.mean(v)), B=B, alpha=ALPHA, seed=SEED)
    print(f'  {ds}: mean Pearson = {point:+.3f} [95% CI: {lo:+.3f}, {hi:+.3f}], n_models={len(sub)}')

Loaded SCTS canonical: 18000 rows
✓ Saved bootstrap_cis_scts_pearson.csv (18 rows, 0.1 min)

Per-dataset Pearson (mean across 6 models, with bootstrap CI for the mean across models):
  nsl_kdd_v2: mean Pearson = +0.055 [95% CI: -0.031, +0.145], n_models=6
  unsw_nb15_v2: mean Pearson = +0.465 [95% CI: +0.438, +0.491], n_models=6
  cic_ids2017_v2: mean Pearson = +0.318 [95% CI: +0.233, +0.408], n_models=6


## 7. Target 5: SCTS Normal-vs-U2R per-class gap

In [8]:
# Aggregate gap across all 18 models, stratified bootstrap by true_class
# The gap = mean SCTS for Normal samples - mean SCTS for U2R samples

# Filter to Normal and U2R only
mask = df_scts['true_class'].isin([0, 4])
sub = df_scts[mask][['true_class', 'scts']].copy()

def gap_stat(values, strata):
    normal_mean = values[strata == 0].mean()
    u2r_mean = values[strata == 4].mean()
    return float(normal_mean - u2r_mean)

point, lo, hi = bootstrap_ci_stratified(
    sub['scts'].values, sub['true_class'].values, gap_stat,
    B=B, alpha=ALPHA, seed=SEED
)
print(f'Normal-vs-U2R SCTS gap (aggregate across all 18 models):')
print(f'  Point: {point:.2f}, 95% CI: [{lo:.2f}, {hi:.2f}]')
print(f'  n_Normal = {(sub["true_class"] == 0).sum()}, n_U2R = {(sub["true_class"] == 4).sum()}')

# Also per-dataset for context
gap_records = [{'scope': 'aggregate', 'dataset': 'all', 'point': point, 'lo_95': lo, 'hi_95': hi,
                'n_normal': int((sub['true_class'] == 0).sum()), 'n_u2r': int((sub['true_class'] == 4).sum())}]

for ds in DATASETS:
    ds_sub = df_scts[(df_scts['dataset'] == ds) & df_scts['true_class'].isin([0, 4])][['true_class', 'scts']]
    if (ds_sub['true_class'] == 0).sum() < 5 or (ds_sub['true_class'] == 4).sum() < 5:
        continue
    point_ds, lo_ds, hi_ds = bootstrap_ci_stratified(
        ds_sub['scts'].values, ds_sub['true_class'].values, gap_stat,
        B=B, alpha=ALPHA, seed=SEED
    )
    gap_records.append({
        'scope': 'per_dataset', 'dataset': ds,
        'point': point_ds, 'lo_95': lo_ds, 'hi_95': hi_ds,
        'n_normal': int((ds_sub['true_class'] == 0).sum()),
        'n_u2r': int((ds_sub['true_class'] == 4).sum()),
    })
    print(f'  {ds}: gap = {point_ds:.2f}, 95% CI: [{lo_ds:.2f}, {hi_ds:.2f}]')

df_gap = pd.DataFrame(gap_records)
df_gap.to_csv(TABLES_DIR / 'bootstrap_cis_scts_class_gap.csv', index=False)
print(f'\n✓ Saved bootstrap_cis_scts_class_gap.csv ({len(df_gap)} rows)')

Normal-vs-U2R SCTS gap (aggregate across all 18 models):
  Point: 12.03, 95% CI: [11.19, 13.03]
  n_Normal = 4890, n_U2R = 1644
  nsl_kdd_v2: gap = 6.30, 95% CI: [4.84, 7.92]
  unsw_nb15_v2: gap = 10.54, 95% CI: [9.17, 11.92]
  cic_ids2017_v2: gap = 29.72, 95% CI: [17.39, 41.76]

✓ Saved bootstrap_cis_scts_class_gap.csv (4 rows)


## 8. Target 6: GREEN vs RED Pearson gap (the health flag claim)

In [ ]:
df_health = pd.read_csv(TABLES_DIR / 'scts_v2_canonical_with_health.csv')
print(f'Loaded augmented SCTS: {len(df_health)} rows')

t0 = time.time()

# Per-flag Pearson with CI: per (dataset, model) within each flag group, bootstrap Pearson
flag_records = []

for flag in ['green', 'amber', 'red']:
    flag_sub = df_health[df_health['calib_health'] == flag]
    if len(flag_sub) == 0:
        continue
    for (ds, model), g in flag_sub.groupby(['dataset', 'model']):
        scts = g['scts'].values
        correct = g['correct'].values.astype(float)
        if scts.std() < 1e-9 or correct.std() < 1e-9 or len(scts) < 10:
            continue

        def pearson_fn(s, c):
            if s.std() < 1e-9 or c.std() < 1e-9:
                return np.nan
            return float(np.corrcoef(s, c)[0, 1])

        point, lo, hi = bootstrap_ci_paired(scts, correct, pearson_fn, B=B, alpha=ALPHA, seed=SEED)
        flag_records.append({
            'flag': flag, 'dataset': ds, 'model': model,
            'pearson': point, 'lo_95': lo, 'hi_95': hi, 'n': len(scts),
        })

df_flag_pearson = pd.DataFrame(flag_records)
df_flag_pearson.to_csv(TABLES_DIR / 'bootstrap_cis_flag_pearson.csv', index=False)
print(f'✓ Saved bootstrap_cis_flag_pearson.csv ({len(df_flag_pearson)} rows, {(time.time()-t0)/60:.1f} min)')

# Per-flag mean Pearson with CI on the per-model means
print('\nPer-flag Pearson (mean across valid models, with CI on the mean):')
flag_summary = []
for flag in ['green', 'amber', 'red']:
    sub = df_flag_pearson[df_flag_pearson['flag'] == flag]
    sub = sub.dropna(subset=['pearson'])
    if len(sub) < 2:
        continue
    point, lo, hi = bootstrap_ci(sub['pearson'].values, lambda v: float(np.mean(v)), B=B, alpha=ALPHA, seed=SEED)
    flag_summary.append({
        'flag': flag, 'n_models': len(sub),
        'mean_pearson': point, 'lo_95': lo, 'hi_95': hi,
    })
    print(f'  {flag.upper()}: mean Pearson = {point:+.3f} [95% CI: {lo:+.3f}, {hi:+.3f}], n_models={len(sub)}')

# The CRITICAL test: GREEN minus RED gap with CI
green_pearsons = df_flag_pearson[df_flag_pearson['flag'] == 'green']['pearson'].dropna().values
red_pearsons = df_flag_pearson[df_flag_pearson['flag'] == 'red']['pearson'].dropna().values

# Independent (not paired) bootstrap on the gap of means
rng = np.random.default_rng(SEED)
gap_boot = np.empty(B)
for b in range(B):
    g_idx = rng.integers(0, len(green_pearsons), size=len(green_pearsons))
    r_idx = rng.integers(0, len(red_pearsons), size=len(red_pearsons))
    gap_boot[b] = green_pearsons[g_idx].mean() - red_pearsons[r_idx].mean()
gap_point = green_pearsons.mean() - red_pearsons.mean()
gap_lo = np.quantile(gap_boot, ALPHA / 2)
gap_hi = np.quantile(gap_boot, 1 - ALPHA / 2)

print(f'\nGREEN minus RED Pearson gap:')
print(f'  Point: {gap_point:+.3f}, 95% CI: [{gap_lo:+.3f}, {gap_hi:+.3f}]')
print(f'  Significantly greater than zero: {gap_lo > 0}')

flag_summary.append({
    'flag': 'green_minus_red_gap', 'n_models': f'{len(green_pearsons)} vs {len(red_pearsons)}',
    'mean_pearson': float(gap_point), 'lo_95': float(gap_lo), 'hi_95': float(gap_hi),
})
df_flag_summary = pd.DataFrame(flag_summary)
df_flag_summary.to_csv(TABLES_DIR / 'bootstrap_cis_flag_pearson_summary.csv', index=False)
print(f'✓ Saved bootstrap_cis_flag_pearson_summary.csv')

Loaded augmented SCTS: 18000 rows


## 9. Target 7: NSL R2L RED-flag rate (Wilson binomial CI)

In [ ]:
nsl_r2l = df_health[(df_health['dataset'] == 'nsl_kdd_v2') & (df_health['true_class'] == 3)]
n_total = len(nsl_r2l)
n_red = (nsl_r2l['calib_health'] == 'red').sum()
n_green = (nsl_r2l['calib_health'] == 'green').sum()
n_amber = (nsl_r2l['calib_health'] == 'amber').sum()

p, lo, hi = wilson_ci(n_red, n_total, alpha=ALPHA)
print(f'NSL R2L true-class samples (n={n_total}):')
print(f'  RED-flagged: {n_red}/{n_total} = {p*100:.1f}% [95% Wilson CI: {lo*100:.1f}%, {hi*100:.1f}%]')
print(f'  GREEN-flagged: {n_green}/{n_total} = {n_green/n_total*100:.1f}%')
print(f'  AMBER-flagged: {n_amber}/{n_total} = {n_amber/n_total*100:.1f}%')

r2l_records = [
    {'flag': 'red', 'successes': int(n_red), 'n': int(n_total),
     'proportion': float(p), 'lo_95': float(lo), 'hi_95': float(hi), 'method': 'wilson'},
]
# Also do GREEN and AMBER for completeness
for f_name, count in [('green', n_green), ('amber', n_amber)]:
    p_f, lo_f, hi_f = wilson_ci(count, n_total, alpha=ALPHA)
    r2l_records.append({
        'flag': f_name, 'successes': int(count), 'n': int(n_total),
        'proportion': float(p_f), 'lo_95': float(lo_f), 'hi_95': float(hi_f), 'method': 'wilson',
    })

df_r2l = pd.DataFrame(r2l_records)
df_r2l.to_csv(TABLES_DIR / 'bootstrap_cis_r2l_flag_rate.csv', index=False)
print(f'\n✓ Saved bootstrap_cis_r2l_flag_rate.csv')

## 10. Master summary JSON + commit

In [ ]:
summary = {
    'timestamp': datetime.now().isoformat(),
    'protocol': {
        'B': B,
        'alpha': ALPHA,
        'ci_level': '95%',
        'seed': SEED,
        'method': 'percentile_bootstrap (Wilson for binomial)',
    },
    'targets': {
        'target_1_calibration': {
            'description': 'Brier per-class and ECE per-model with CIs',
            'output': 'bootstrap_cis_calibration.csv',
            'n_rows': len(df_calib),
        },
        'target_2_krishna': {
            'description': 'Mean rank correlation CIs per (dataset, pair, K)',
            'output': 'bootstrap_cis_krishna.csv',
            'n_rows': len(df_krishna_cis),
        },
        'target_3_stability': {
            'description': 'Mean Jaccard per (dataset, model) + DNN advantage CI',
            'outputs': ['bootstrap_cis_stability.csv', 'bootstrap_cis_stability_advantage.csv'],
            'dnn_advantage_point_count': int(df_advantage['dnn_advantage'].sum()),
            'dnn_advantage_significant_count': int(df_advantage['significant_advantage'].sum()),
            'total_comparisons': len(df_advantage),
        },
        'target_4_scts_pearson': {
            'description': 'SCTS-vs-correctness Pearson per (dataset, model) with CI',
            'output': 'bootstrap_cis_scts_pearson.csv',
            'n_rows': len(df_pearson),
        },
        'target_5_class_gap': {
            'description': 'Normal-vs-U2R SCTS gap with CI',
            'output': 'bootstrap_cis_scts_class_gap.csv',
            'aggregate_gap': {
                'point': float(df_gap[df_gap['scope'] == 'aggregate']['point'].iloc[0]),
                'lo_95': float(df_gap[df_gap['scope'] == 'aggregate']['lo_95'].iloc[0]),
                'hi_95': float(df_gap[df_gap['scope'] == 'aggregate']['hi_95'].iloc[0]),
            },
        },
        'target_6_flag_pearson_gap': {
            'description': 'GREEN vs RED Pearson gap with CI',
            'outputs': ['bootstrap_cis_flag_pearson.csv', 'bootstrap_cis_flag_pearson_summary.csv'],
            'green_minus_red': {
                'point': float(gap_point),
                'lo_95': float(gap_lo),
                'hi_95': float(gap_hi),
                'significantly_positive': bool(gap_lo > 0),
            },
        },
        'target_7_r2l_red_rate': {
            'description': 'NSL R2L true-class RED-flag rate, Wilson CI',
            'output': 'bootstrap_cis_r2l_flag_rate.csv',
            'red_rate': {
                'point': float(p),
                'lo_95': float(lo),
                'hi_95': float(hi),
                'n_total': int(n_total),
                'n_red': int(n_red),
            },
        },
    },
}

with open(TABLES_DIR / 'bootstrap_cis_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('✓ Saved bootstrap_cis_summary.json')
print('\nSummary:')
print(json.dumps(summary, indent=2)[:3000])
print('... (truncated)')

In [ ]:
os.chdir(REPO)
!git add notebooks/08_bootstrap_cis.ipynb
!git add results/tables/bootstrap_cis_calibration.csv
!git add results/tables/bootstrap_cis_krishna.csv
!git add results/tables/bootstrap_cis_stability.csv
!git add results/tables/bootstrap_cis_stability_advantage.csv
!git add results/tables/bootstrap_cis_scts_pearson.csv
!git add results/tables/bootstrap_cis_scts_class_gap.csv
!git add results/tables/bootstrap_cis_flag_pearson.csv
!git add results/tables/bootstrap_cis_flag_pearson_summary.csv
!git add results/tables/bootstrap_cis_r2l_flag_rate.csv
!git add results/tables/bootstrap_cis_summary.json
!git status --short
!git commit -m 'Notebook 08: Bootstrap CIs for 7 Day 4 targets (B=1000 percentile bootstrap + Wilson binomial). Calibration Brier/ECE, Krishna RC, DNN stability advantage, SCTS Mondrian Pearson per dataset, Normal-vs-U2R gap, GREEN-vs-RED Pearson gap, NSL R2L RED-flag rate.'
!git push origin main